# Day 3 · 변경 라인에 근거를 남기는 코드 리뷰 Agent

화면을 따라 실행하되, 결과를 자동 게시하지 않습니다. 모든 외부 쓰기는 dry-run과 사람 승인을 먼저 거칩니다.

In [ ]:
# 최초 1회 설치. 이미 설치했다면 빠르게 완료됩니다.
%pip install -q -r ../../requirements-day1.txt
# STT 실습을 실제 음성으로 실행할 때만 다음 줄의 주석을 해제합니다.
# %pip install -q -r ../../requirements-stt-optional.txt

In [ ]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({"workspace": str(ROOT), "python": sys.version.split()[0]})

## 1. unified diff를 읽고 추가 라인 번호를 복원합니다

In [ ]:
from src.course_services.review_service import parse_unified_diff, run_review_service

diff_path = ROOT / "data/day3_review_cases/unsafe_pr.diff"
diff_text = diff_path.read_text(encoding="utf-8")
parsed = parse_unified_diff(diff_text)
print(parsed.changed_paths)
print([(line.line, line.text) for line in parsed.added_lines])

## 2. 결정론적 baseline으로 고위험 finding을 만듭니다

In [ ]:
review = run_review_service(diff_text)
print(json.dumps(review, ensure_ascii=False, indent=2))
assert all(finding["line"] in {line.line for line in parsed.added_lines} for finding in review["findings"])
assert review["automatic_publish"] is False

## 3. Codex에게 맡길 작업도 scope와 test 계약부터 씁니다

In [ ]:
from src.course_services.codex_harness import CodexTaskSpec, render_codex_task

spec = CodexTaskSpec(
    objective="새 review rule 하나와 정상·실패 test를 추가한다.",
    allowed_paths=("src/course_services", "tests"),
    acceptance_tests=("python -m pytest -q tests/test_course_services.py",),
)
print(render_codex_task(spec))

## 완료 확인

- Day 3 결과 JSON을 확인했습니다.
- 실패 경로가 traceback 대신 `error_code`로 남는지 확인했습니다.
- 외부 쓰기와 자동 메일이 발생하지 않았음을 확인했습니다.
- 변경한 코드는 diff와 test 결과를 사람이 검토합니다.